# CASTalign Testground

Comprehensive alignment workflow for 2D histology slices and 3D volumes.

## Workflow Overview
1. **Load graph** with in-vivo reference + subslices
2. **Select slice** via dropdown widget
3. **Align** using one of four modes:
   - Mode A: Padded, 3D as base (point-based, RAM-heavy)
   - Mode B: No padding, 3D as base (parametric only, low RAM)
   - Mode C: No padding, 2D as base / 3D movable (point-based, low RAM)
   - Mode D: Same-dimension 3D volumes (adapted from qy47's workflow)
4. **Verify** alignment with Pearson correlation + napari overlay
5. **Refine** with Triangulation2D for edge warping
6. **Export** warped images as TIFF
7. **Visualize** all aligned slices with GraphViewer

---

## 1. Imports & Setup

In [ ]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================
%reset -f

import sys
import gc
from pathlib import Path
import numpy as np
import imageio.v2 as imageio
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import pearsonr

# =============================================================================
# PATH SETUP
# =============================================================================
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # cell2cell-alignment/

# Add project root to path for local_config and utilities
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# =============================================================================
# LOCAL CONFIG (user-specific paths)
# =============================================================================
try:
    from local_config import CASTALIGN_ROOT, OUTPUT_ROOT
except ImportError:
    raise ImportError(
        "local_config.py not found.\n"
        "Copy local_config.example.py to local_config.py and fill in your paths:\n"
        "    cp local_config.example.py local_config.py"
    )

CASTALIGN_ROOT = Path(CASTALIGN_ROOT)
if str(CASTALIGN_ROOT) not in sys.path:
    sys.path.insert(0, str(CASTALIGN_ROOT))

print(f"CASTalign: {CASTALIGN_ROOT}")

# =============================================================================
# CASTALIGN IMPORTS
# =============================================================================
try:
    import castalign as ca
    import castalign.gui as ca_gui
    from castalign.gui import GraphViewer
    from castalign import ndarray_shifted
    import castalign.utils as ca_utils
    print("Imported castalign")
except ImportError:
    # Fallback to linestuffup if castalign not available
    import linestuffup as ca
    import linestuffup.gui as ca_gui
    from linestuffup import ndarray_shifted
    import linestuffup.utils as ca_utils
    try:
        from linestuffup.gui import GraphViewer
    except ImportError:
        GraphViewer = None
        print("Warning: GraphViewer not available in this version")
    print("Imported linestuffup (castalign fallback)")

# =============================================================================
# PROJECT UTILITIES
# =============================================================================
from utilities.graph_viz_fix import patch_castalign_visualise
patch_castalign_visualise()

# =============================================================================
# JUPYTER MAGIC
# =============================================================================
%load_ext autoreload
%autoreload 2

# Disable label auto-detection
ca_utils.image_is_label = lambda img: False

print()
print("="*60)
print("CASTALIGN TESTGROUND")
print("="*60)
print("Imports complete")
print("GraphViz patched (SVG output)")
print("Auto-reload enabled")
print("="*60)

## 2. Path Configuration

In [ ]:
# =============================================================================
# PATH CONFIGURATION (from local_config.py)
# =============================================================================
#
# WARNING: SQLite over Samba (Mac accessing server files)
# --------------------------------------------------------
# The graph .db files are SQLite databases. Writing SQLite over network mounts
# (Samba/SMB) can sometimes cause issues:
#   - "database is locked" errors
#   - Corruption errors on save
#   - Hanging during graph.save()
#
# If you encounter these issues, consider:
#   1. Copy graph locally, align, copy back
#   2. Run alignment directly on the server (SSH + X11 forwarding)
#
# Initial file loading (200MB in vivo stack) may take 30-60 seconds over network,
# but once in RAM, napari will be smooth.
# =============================================================================

# =============================================================================
# DATA PATHS (derived from OUTPUT_ROOT set in local_config.py)
# =============================================================================
OUTPUT_ROOT = Path(OUTPUT_ROOT)

# In-vivo reference
INVIVO_STACK_PATH = OUTPUT_ROOT / "in_vivo_flip_corrected" / "JH302_1x_ch2_flipped.tiff"

# Subslice directory (anisotropic cell mask overlays)
ANISO_SUBSLICE_DIR = OUTPUT_ROOT / "mScarlet_cellmask_subslice" / "threshold_0.30_cellmask_0.50_anisotropic"

# Graph paths (read AND write to server)
GRAPH_DIR = OUTPUT_ROOT / "linestuffup_output"
GRAPH_PATH = GRAPH_DIR / "castalign_test.db"

# Warped image output directory
WARPED_OUTPUT_DIR = OUTPUT_ROOT / "warped_output"

# Same-dimension alignment graph directory (optional, for Mode D)
SAME_DIM_GRAPH_DIR = OUTPUT_ROOT / "same_dim_output"

# =============================================================================
# PATH VERIFICATION
# =============================================================================
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print()
print("Path verification:")
print(f"  In-vivo stack: {'OK' if INVIVO_STACK_PATH.exists() else 'NOT FOUND'} {INVIVO_STACK_PATH.name}")
print(f"  Subslice dir:  {'OK' if ANISO_SUBSLICE_DIR.exists() else 'NOT FOUND'} {ANISO_SUBSLICE_DIR.name}")
print(f"  Graph file:    {'OK' if GRAPH_PATH.exists() else 'WILL BE CREATED'} {GRAPH_PATH.name}")
print(f"  Warped output: {'OK' if WARPED_OUTPUT_DIR.exists() else 'WILL BE CREATED'} {WARPED_OUTPUT_DIR.name}")
print(f"  Same-dim dir:  {'OK' if SAME_DIM_GRAPH_DIR.exists() else 'WILL BE CREATED'} {SAME_DIM_GRAPH_DIR.name}")

if ANISO_SUBSLICE_DIR.exists():
    n_slices = len(list(ANISO_SUBSLICE_DIR.glob("*.tif")))
    print(f"\n  Found {n_slices} subslice files")

## 3. Load Graph

In [ ]:
# =============================================================================
# LOAD GRAPH
# =============================================================================

if not GRAPH_PATH.exists():
    raise FileNotFoundError(f"Graph not found: {GRAPH_PATH}\nRun subslice_graph_builder.py to build it first.")

g = ca.Graph.load(str(GRAPH_PATH))

# Get slice info
all_nodes = list(g.nodes)
invivo_node = "invivo_ref"

# Find slice nodes (all nodes except invivo_ref)
# Node naming: slice{N}_subslice_mScarlet_cellmask
slice_nodes = sorted([n for n in all_nodes if n != invivo_node])

# Check which slices are already aligned
aligned_slices = []
unaligned_slices = []
for s in slice_nodes:
    if s in g.edges and invivo_node in g.edges[s]:
        aligned_slices.append(s)
    else:
        unaligned_slices.append(s)

print("="*60)
print("GRAPH LOADED")
print("="*60)
print(f"Graph: {g.name}")
print(f"Total nodes: {len(all_nodes)}")
print(f"Subslices: {len(slice_nodes)}")
print(f"  Aligned: {len(aligned_slices)}")
print(f"  Unaligned: {len(unaligned_slices)}")
print("="*60)

## 4. Slice Selector

Use the dropdown to select which slice to align. Green = aligned, Red = unaligned.

In [ ]:
# =============================================================================
# SLICE SELECTOR WIDGET
# =============================================================================
import re

def extract_slice_number(node_name):
    """Extract slice number from node name like 'slice10_subslice_mScarlet_cellmask'."""
    match = re.match(r'slice(\d+)', node_name)
    return int(match.group(1)) if match else 0

# Sort slice nodes by number
slice_nodes = sorted(slice_nodes, key=extract_slice_number)

# Create options with alignment status
def get_slice_options():
    """Generate dropdown options with alignment status and transform type."""
    options = []
    for s in slice_nodes:
        is_aligned = s in aligned_slices
        num = extract_slice_number(s)
        if is_aligned:
            t = g.get_transform(s, invivo_node)
            tname = type(t).__name__
            options.append((f"OK [{tname}] Slice {num}", s))
        else:
            options.append((f"o  Slice {num}", s))
    return options

# Create widget
slice_dropdown = widgets.Dropdown(
    options=get_slice_options(),
    description="Select:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="350px")
)

# Info display
info_output = widgets.Output()

def show_slice_info(node):
    """Display info for a slice node."""
    with info_output:
        clear_output(wait=True)
        img = g.get_image(node)
        is_aligned = node in aligned_slices
        
        print(f"Selected: {node}")
        print(f"Shape: {img.shape}")
        print(f"Status: {'Aligned' if is_aligned else 'Not aligned'}")
        
        if is_aligned:
            t = g.get_transform(node, invivo_node)
            print(f"Transform: {type(t).__name__}")

def on_slice_change(change):
    """Update info when slice selection changes."""
    show_slice_info(change["new"])

# Attach observer FIRST
slice_dropdown.observe(on_slice_change, names="value")

# Display widget
print("Select a slice to align:")
print("  OK [TransformType] = already aligned")
print("  o  = not yet aligned")
print()
display(widgets.HBox([slice_dropdown, info_output]))

# Show initial info (observer won't fire since value hasn't changed)
show_slice_info(slice_dropdown.value)

def get_selected_slice():
    """Get currently selected slice name."""
    return slice_dropdown.value

## 5. Alignment Utilities

Helper functions used by all alignment modes.

In [ ]:
# =============================================================================
# ALIGNMENT UTILITIES
# =============================================================================

def get_initial_transform_3d_base():
    """
    Initial transform for 3D-as-base mode.
    
    Applies:
    - xrotate=90: Maps ex-vivo Y -> in-vivo Z (dorsal stays on top)
    - FlipX: Reverses left-right to match in-vivo orientation
    """
    rotation = ca.TranslateRotateFixed(xrotate=90)
    flip = ca.FlipFixed(x=True)
    return rotation + flip


def get_initial_transform_3d_movable():
    """
    Initial transform for 3D-as-movable mode.
    
    TODO: This needs adjustment - the orientation flip may differ
    when the 3D volume is movable. Return to this after testing.
    """
    return ca.Identity()


def load_and_pad_slice(slice_name, pad_z=25):
    """
    Load a slice from the graph and pad it in z for point-based transforms.
    
    Parameters
    ----------
    slice_name : str
        Name of the slice node in the graph
    pad_z : int
        Number of blank slices above and below (total z = 2*pad_z + 1)
    
    Returns
    -------
    padded : np.ndarray
        Padded volume with original slice at z=pad_z
    """
    slice_img = g.get_image(slice_name)
    padding_value = slice_img.mean() * 0.90
    padded = np.full((2 * pad_z + 1, *slice_img.shape[1:]), padding_value, dtype=slice_img.dtype)
    padded[pad_z] = slice_img[0]
    return padded


def compute_alignment_quality(fixed_img, movable_img, transform):
    """
    Apply a transform and compute Pearson correlation as alignment quality metric.
    
    Parameters
    ----------
    fixed_img : np.ndarray
        The fixed/reference image
    movable_img : np.ndarray
        The movable image (pre-transform)
    transform : Transform
        The alignment transform (movable -> fixed direction)
    
    Returns
    -------
    r : float
        Pearson correlation coefficient between overlapping regions
    warped : np.ndarray
        The transformed movable image in fixed space
    """
    warped = transform.transform_image(
        movable_img,
        output_size=list(fixed_img.shape),
    )
    warped_arr = np.asarray(warped).astype(np.float64)
    fixed_arr = np.asarray(fixed_img).astype(np.float64)
    
    # Mask to overlapping non-zero region
    mask = (warped_arr > 0) & (fixed_arr > 0)
    if mask.sum() < 10:
        print("Warning: fewer than 10 overlapping non-zero voxels")
        return 0.0, warped_arr
    
    r, _ = pearsonr(fixed_arr[mask], warped_arr[mask])
    return r, warped_arr


def export_warped_image(warped_array, output_path, dtype=None):
    """
    Save a warped volume as a multi-page TIFF.
    
    Parameters
    ----------
    warped_array : np.ndarray
        3D array (Z, Y, X) to save
    output_path : str or Path
        Output file path (.tif or .tiff)
    dtype : numpy dtype, optional
        Cast to this dtype before saving. If None, keeps original.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    arr = np.asarray(warped_array)
    if dtype is not None:
        arr = arr.astype(dtype)
    
    imageio.mimwrite(str(output_path), [arr[z] for z in range(arr.shape[0])])
    print(f"Saved: {output_path}")
    print(f"  Shape: {arr.shape}, dtype: {arr.dtype}")
    print(f"  Size: {output_path.stat().st_size / 1e6:.1f} MB")


def save_alignment(slice_name, transform):
    """
    Save alignment to graph and disk.
    
    Parameters
    ----------
    slice_name : str
        Name of the slice node
    transform : Transform
        The fitted transform (slice -> invivo_ref direction)
    """
    global aligned_slices, unaligned_slices
    
    # Check if edge already exists to decide update flag
    edge_exists = (slice_name, invivo_node) in g
    g.add_edge(slice_name, invivo_node, transform, update=edge_exists)
    
    # Save to disk
    g.save()
    
    # Update tracking lists
    if slice_name not in aligned_slices:
        aligned_slices.append(slice_name)
    if slice_name in unaligned_slices:
        unaligned_slices.remove(slice_name)
    
    # Update dropdown options
    slice_dropdown.options = get_slice_options()
    
    print(f"\nSaved: {slice_name} -> {invivo_node}")
    print(f"  Transform: {type(transform).__name__}")
    print(f"  Graph saved to: {GRAPH_PATH.name}")


def get_previous_transform(slice_name):
    """
    Get transform from a nearby aligned slice for rotation carry-forward.
    
    Returns the transform from the closest lower-numbered aligned slice,
    or None if no previous slices are aligned.
    """
    current_num = extract_slice_number(slice_name)
    if current_num == 0:
        return None
    
    best_match = None
    best_diff = float("inf")
    
    for s in aligned_slices:
        num = extract_slice_number(s)
        if num > 0 and num < current_num:
            diff = current_num - num
            if diff < best_diff:
                best_diff = diff
                best_match = s
    
    if best_match:
        return g.get_transform(best_match, invivo_node)
    return None


print("Alignment utilities loaded:")
print("  - get_initial_transform_3d_base()")
print("  - get_initial_transform_3d_movable()")
print("  - load_and_pad_slice(slice_name, pad_z=25)")
print("  - compute_alignment_quality(fixed, movable, transform)")
print("  - export_warped_image(warped, output_path, dtype=None)")
print("  - save_alignment(slice_name, transform)")
print("  - get_previous_transform(slice_name)")

---

## Transform Reference & Geometry Guide

### Transform Keybindings

`align_interactive()` prompts you to choose a transform. These are the available keys:

| Key | Transform Class | Type | Best for |
|-----|----------------|------|----------|
| `t` | TranslateRotateFixed | Parametric (sliders) | Quick slider-based rigid alignment |
| `r` | TranslateRotateRescaleFixed | Parametric (sliders) | Slider-based with scaling |
| `T` | TranslateRotate | Point-based | Precise rigid alignment via landmarks |
| `R` | TranslateRotateRescale | Point-based | Full affine via landmarks (cake) |
| `P` | TranslateRotateRescaleByPlane | Point-based | Pancake/rice-paper with scaling |
| `N` | Triangulation2D | Point-based | Nonlinear refinement for pancake/rice-paper (projected) |
| `V` | Triangulation | Point-based | Nonlinear refinement for cake (3D) |

### Modifier Keys (during GUI)

| Key | Action |
|-----|--------|
| `e` | Edit the previous transform (re-open GUI with current params) |
| `z` | Remove the most recent transform layer |
| `u` | Undo — revert to previous transform state |
| `f` | Flip movable along z axis |
| `d` | Toggle reference images on/off |
| `x_` | Extend previous point-based transform with a new one (e.g., `xN` = add Triangulation2D) |
| `c_` | Convert previous point-based transform type (e.g., `cR` = convert to TranslateRotateRescale) |
| `s` | Save transform to graph (in memory only) |
| `S` | Save transform to graph AND write .db to disk |
| `q` | Quit — return the current transform |
| `v` | View only — opens GUI but doesn't change the transform |

### Geometry Guide

The choice of transform depends on the **relative dimensionality** of your images:

- **Cake** (3D vs 3D, same dimensions): Both volumes have z-extent. Use `R` (affine) then `V` (nonlinear 3D triangulation).
- **Pancake** (3D vs 3D, different z-extent): One volume has less z range. Use `P` (plane-based rescale) then `N` (projected triangulation).
- **Rice paper** (2D slice vs 3D volume): The 2D slice has only 1 z-plane. **Must pad** to enable point-based transforms. Use `T` (rigid) then `N` (projected triangulation).

**Padding rationale:** Point-based transforms need at least a few z-slices to fit 3D correspondences. We pad the 1-voxel slice to `2*PAD_Z+1` slices (default 51) with the original at the center. The padding value is 90% of the slice mean to avoid confusing the transform fitter.

---

## 6. Alignment Mode A: Padded, 3D as Base

**Use when:** You need point-based transforms (T, R, N, V, P) on rice-paper slices.

**How it works:** 
- Pads the 1-voxel slice to give it z-extent, enabling point-based fitting
- 3D in-vivo stack is the base (fixed) image

**RAM usage:** ~400-500 MB

**Transform type:** TranslateRotate (point-based)

**Controls:**
- Click corresponding points on base and movable
- Press 'q' to quit and save

In [ ]:
# =============================================================================
# MODE A: PADDED, 3D AS BASE
# =============================================================================

# Get selected slice
slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode: Padded, 3D as base")
print()

# Load and pad slice
PAD_Z = 25
padded = load_and_pad_slice(slice_name, pad_z=PAD_Z)
invivo_img = g.get_image(invivo_node)

print(f"Padded shape: {padded.shape}")
print(f"In-vivo shape: {invivo_img.shape}")
print(f"Memory: {padded.nbytes / 1e6:.1f} MB")

# Get initial transform (or carry forward from previous slice)
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform
    print(f"\nUsing transform from previous aligned slice")
else:
    initial_transform = get_initial_transform_3d_base()
    print(f"\nUsing default initial transform")

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print("Movable: padded 2D slice (green)")
print("Base: 3D in-vivo stack (magenta)")
print("Press 'q' to quit when done")
print()

transform = ca_gui.align_interactive(
    nodes_movable=padded,
    nodes_fixed=invivo_img,
    transform=initial_transform,
    graph=g
)

# Auto-save
save_alignment(slice_name, transform)

# Cleanup
del padded
gc.collect()

---

## 7. Alignment Mode B: No Padding, 3D as Base (Parametric)

**Use when:** You only need slider-based alignment, no keypoints.

**How it works:** 
- Uses the raw 1-voxel slice directly (no padding)
- Parametric (slider-based) transforms only

**RAM usage:** Minimal (~50 MB for slice + in-vivo)

**Transform type:** TranslateRotateFixed (slider-based)

**Controls:**
- Use sliders to adjust rotation and translation
- Press 'q' to quit and save

**Note:** No point clicking in this mode. For point-based, use Mode A or C.

In [ ]:
# =============================================================================
# MODE B: NO PADDING, 3D AS BASE (PARAMETRIC ONLY)
# =============================================================================

# Get selected slice
slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode: No padding, 3D as base (parametric)")
print()

# Load images (no padding)
slice_img = g.get_image(slice_name)
invivo_img = g.get_image(invivo_node)

print(f"Slice shape: {slice_img.shape}")
print(f"In-vivo shape: {invivo_img.shape}")
print(f"Memory: {slice_img.nbytes / 1e6:.1f} MB (no padding)")

# Get initial transform
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform
    print(f"\nUsing transform from previous aligned slice")
else:
    initial_transform = get_initial_transform_3d_base()
    print(f"\nUsing default initial transform")

# Warning
print("\n" + "!"*60)
print("NOTE: No padding - point-based transforms (T, R, P) may not work well")
print("Use parametric transforms (t, r) or switch to Mode A for point-based")
print("!"*60)

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print("Movable: 2D slice (green)")
print("Base: 3D in-vivo stack (magenta)")
print("Press 'q' to quit when done")
print()

transform = ca_gui.align_interactive(
    nodes_movable=slice_img,
    nodes_fixed=invivo_img,
    transform=initial_transform,
    graph=g
)

# Auto-save
save_alignment(slice_name, transform)

---

## 8. Alignment Mode C: Padded 2D as Base (3D Movable)

**Use when:** You want to scroll through the 3D volume to find where the 2D slice fits.

**How it works:** 
- 2D slice is base with padding (PAD_Z=25, same as Mode A)
- 3D volume is movable
- No pre-transforms - manually rotate in napari as needed
- Graph stores both directions automatically

**RAM usage:** ~300-400 MB (in-vivo ~200MB + padded slice ~100MB)

**Benefits:**
- All transforms work (T, R, N, V, P) due to padding
- Can scroll through 3D to find matching z-location
- Full manual control over orientation

In [ ]:
# =============================================================================
# MODE C: PADDED 2D AS BASE (3D MOVABLE)
# =============================================================================

# Get selected slice
slice_name = get_selected_slice()
print(f"Aligning: {slice_name}")
print(f"Mode: Padded 2D as base (3D MOVABLE)")
print()

# Load and pad slice as base, in-vivo as movable (roles swapped!)
PAD_Z = 25
padded_base = load_and_pad_slice(slice_name, pad_z=PAD_Z)
invivo_img = g.get_image(invivo_node)

print(f"Base (2D slice, padded): {padded_base.shape}")
print(f"Movable (3D volume): {invivo_img.shape}")
print(f"Memory: {padded_base.nbytes / 1e6:.1f} MB")

# Initial transform for movable (3D volume) - start with identity
initial_transform = ca.Identity()

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI")
print("="*60)
print("Movable: 3D in-vivo volume (green)")
print("Base: 2D slice (magenta) - padded")
print("Use napari controls to manually rotate/adjust view")
print("Press 'q' to quit when done")
print()

transform_inv = ca_gui.align_interactive(
    nodes_movable=invivo_img,   # 3D as movable
    nodes_fixed=padded_base,    # 2D as base (padded)
    transform=initial_transform,
    graph=g
)

# Save transform (invivo -> slice direction)
edge_exists = (invivo_node, slice_name) in g
g.add_edge(invivo_node, slice_name, transform_inv, update=edge_exists)
g.save()

# Update tracking
if slice_name not in aligned_slices:
    aligned_slices.append(slice_name)
if slice_name in unaligned_slices:
    unaligned_slices.remove(slice_name)
slice_dropdown.options = get_slice_options()

print(f"\nSaved: {invivo_node} -> {slice_name} (bidirectional)")
print(f"  Transform: {type(transform_inv).__name__}")
print(f"  Graph saved to: {GRAPH_PATH.name}")

# Cleanup
del padded_base
gc.collect()

---

## 9. Alignment Mode D: Same-Dimension Volumes (3D vs 3D)

**Use when:** Aligning two 3D volumes that share the same voxel grid (e.g., two imaging sessions of the same region).

**How it works:**
- Both volumes loaded as float32 via `imageio.volread()`
- Separate graph for same-dimension alignments (optional)
- No padding needed — both volumes have z-extent
- Use `R` (affine) then `V` (3D triangulation) for best results

**RAM usage:** Depends on volume size (~200 MB per volume)

**Adapted from:** qy47's `linestuffup_same_dimensions.ipynb` workflow

In [ ]:
# =============================================================================
# MODE D: SAME-DIMENSION VOLUME ALIGNMENT
# =============================================================================

# ---- CONFIGURATION (edit these) ----
FIXED_PATH = Path("/path/to/your/fixed_volume.tif")     # Reference volume
MOVABLE_PATH = Path("/path/to/your/movable_volume.tif")  # Volume to warp
FIXED_NAME = "fixed_ref"
MOVABLE_NAME = "movable_vol"
SAMEDIM_GRAPH_PATH = SAME_DIM_GRAPH_DIR / "same_dim_alignment.db"
LOAD_EXISTING_GRAPH = True  # Set False to create fresh graph
# ------------------------------------

# Load volumes
fixed_raw = imageio.volread(str(FIXED_PATH))
movable_raw = imageio.volread(str(MOVABLE_PATH))
fixed = fixed_raw.astype(np.float32)
movable = movable_raw.astype(np.float32)

print(f"Fixed   shape: {fixed.shape}  dtype: {fixed.dtype}  range: [{fixed.min():.0f}, {fixed.max():.0f}]")
print(f"Movable shape: {movable.shape}  dtype: {movable.dtype}  range: [{movable.min():.0f}, {movable.max():.0f}]")

# Create or load graph
if LOAD_EXISTING_GRAPH and SAMEDIM_GRAPH_PATH.exists():
    g_samedim = ca.Graph.load(str(SAMEDIM_GRAPH_PATH))
    print(f"\nLoaded graph: '{g_samedim.name}' from {SAMEDIM_GRAPH_PATH.name}")
else:
    SAME_DIM_GRAPH_DIR.mkdir(parents=True, exist_ok=True)
    g_samedim = ca.Graph(name="same_dim_alignment")
    g_samedim.add_node(FIXED_NAME, fixed, attach_image=False)
    g_samedim.add_node(MOVABLE_NAME, movable, attach_image=False)
    print(f"\nCreated new graph with nodes: {FIXED_NAME}, {MOVABLE_NAME}")

# Initial transform — Identity for same-dimension volumes
# Uncomment below if you need a different starting point:
# initial_transform = ca.FlipFixed(x=True, xthickness=fixed.shape[2])
# initial_transform = ca.TranslateRotateFixed(xrotate=90)
initial_transform = ca.Identity()

# Launch alignment GUI
print("\n" + "="*60)
print("LAUNCHING GUI — Same-Dimension Alignment")
print("="*60)
print(f"Movable: {MOVABLE_NAME} (green)")
print(f"Fixed:   {FIXED_NAME} (magenta)")
print("Recommended: R (affine) then V (nonlinear 3D triangulation)")
print("Press 'q' to quit when done")
print()

t_samedim = ca_gui.align_interactive(
    nodes_movable=movable,
    nodes_fixed=fixed,
    transform=initial_transform,
    graph=g_samedim
)

# Save edge to graph
edge_exists = (MOVABLE_NAME, FIXED_NAME) in g_samedim
g_samedim.add_edge(MOVABLE_NAME, FIXED_NAME, t_samedim, update=edge_exists)
g_samedim.save(str(SAMEDIM_GRAPH_PATH))

print(f"\nSaved: {MOVABLE_NAME} -> {FIXED_NAME}")
print(f"  Transform: {t_samedim}")
print(f"  Graph: {SAMEDIM_GRAPH_PATH.name}")

---

## 10. Post-Alignment Verification

Applies the current transform, computes Pearson correlation, and opens a napari viewer with three layers:
- **Fixed** (gray) — the reference image
- **Warped** (green) — the transformed movable image
- **Original** (red, hidden) — the untransformed movable for comparison

Works for both subslice (Modes A-C) and same-dimension (Mode D) workflows. Set the variables at the top to match which alignment you want to verify.

In [ ]:
# =============================================================================
# POST-ALIGNMENT VERIFICATION
# =============================================================================
# Choose which alignment to verify by setting these variables.
# Option A: Subslice alignment (Modes A-C)
#   verify_fixed = g.get_image(invivo_node)
#   verify_movable = g.get_image(get_selected_slice())
#   verify_transform = g.get_transform(get_selected_slice(), invivo_node)
# Option B: Same-dimension alignment (Mode D)
#   verify_fixed = fixed
#   verify_movable = movable
#   verify_transform = t_samedim
# =============================================================================

# --- Set these for your case ---
slice_name = get_selected_slice()
verify_fixed = g.get_image(invivo_node)
verify_movable = g.get_image(slice_name)
verify_transform = g.get_transform(slice_name, invivo_node)
# -------------------------------

print(f"Verifying alignment: {slice_name} -> {invivo_node}")
print(f"Transform: {type(verify_transform).__name__}")
print()

# Compute quality
r, warped_arr = compute_alignment_quality(verify_fixed, verify_movable, verify_transform)
print(f"Pearson correlation (overlapping region): r = {r:.4f}")
print()

# Open napari viewer
import napari

viewer = napari.Viewer(title=f"Verification: {slice_name}")
viewer.add_image(np.asarray(verify_fixed), name="fixed", colormap="gray", blending="additive", opacity=0.7)
viewer.add_image(warped_arr, name="warped", colormap="green", blending="additive", opacity=0.5)
viewer.add_image(np.asarray(verify_movable), name="original (hidden)", colormap="red", blending="additive", opacity=0.3, visible=False)

print("Napari viewer opened:")
print("  gray  = fixed reference")
print("  green = warped movable")
print("  red   = original movable (hidden, toggle in layer list)")

---

## 11. Nonlinear Refinement: Triangulation2D

**Use after:** Initial affine alignment (Mode A, B, or C)

**What it does:** Adds nonlinear warping to correct edge distortions.

**How it works:**
- Uses padding (required for point-based Triangulation2D)
- Starts from existing transform, adds nonlinear refinement

**How to use:**
1. Select an already-aligned slice
2. Run this cell
3. Press `N` (projected) or `V` (3D) when prompted
4. Click corresponding points on edges that need warping
5. Press 'q' to quit and save

In [ ]:
# =============================================================================
# NONLINEAR REFINEMENT: TRIANGULATION (requires padding)
# =============================================================================

# Get selected slice
slice_name = get_selected_slice()

# Check if slice is aligned
if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
    print("Run one of the alignment modes first.")
else:
    print(f"Refining: {slice_name}")
    print(f"Mode: Nonlinear refinement (N or V transforms)")
    print()
    
    # Get existing transform
    existing_transform = g.get_transform(slice_name, invivo_node)
    print(f"Existing transform: {type(existing_transform).__name__}")
    
    # Load and pad slice
    PAD_Z = 25
    padded = load_and_pad_slice(slice_name, pad_z=PAD_Z)
    invivo_img = g.get_image(invivo_node)
    
    print(f"Padded shape: {padded.shape}")
    
    # Launch GUI with existing transform
    print("\n" + "="*60)
    print("LAUNCHING GUI FOR REFINEMENT")
    print("="*60)
    print("Starting from existing transform")
    print("Use 'N' for projected triangulation or 'V' for 3D triangulation")
    print("Press 'q' to quit when done")
    print()
    
    refined_transform = ca_gui.align_interactive(
        nodes_movable=padded,
        nodes_fixed=invivo_img,
        transform=existing_transform,
        graph=g
    )
    
    # Auto-save
    save_alignment(slice_name, refined_transform)
    
    # Cleanup
    del padded
    gc.collect()

---

## 12. Warped Image Export

Save a warped volume as a multi-page TIFF. Uses the `warped_arr` from the verification cell if available, or recomputes from the graph.

In [ ]:
# =============================================================================
# WARPED IMAGE EXPORT
# =============================================================================

# Which slice to export
slice_name = get_selected_slice()

# Reuse warped_arr from verification cell if it exists and matches
try:
    assert warped_arr is not None
    print(f"Reusing warped array from verification cell")
except (NameError, AssertionError):
    # Recompute
    print(f"Computing warped image for: {slice_name}")
    t = g.get_transform(slice_name, invivo_node)
    invivo_img = g.get_image(invivo_node)
    slice_img = g.get_image(slice_name)
    _, warped_arr = compute_alignment_quality(invivo_img, slice_img, t)

# Export
output_path = WARPED_OUTPUT_DIR / f"{slice_name}_warped.tiff"
export_warped_image(warped_arr, output_path, dtype=np.float32)

---

## 13. Graph Reload Verification

Validates that the graph .db file correctly persists transforms through SQLite serialization. Saves the graph, reloads it, re-applies the transform, and checks that the output matches. Important for Samba mounts where SQLite can be flaky.

In [ ]:
# =============================================================================
# GRAPH RELOAD VERIFICATION
# =============================================================================

slice_name = get_selected_slice()

if slice_name not in aligned_slices:
    print(f"ERROR: {slice_name} is not aligned yet!")
else:
    # Step 1: Get current transform output
    print(f"Verifying graph persistence for: {slice_name}")
    t_original = g.get_transform(slice_name, invivo_node)
    slice_img = g.get_image(slice_name)
    invivo_img = g.get_image(invivo_node)
    
    warped_original = np.asarray(t_original.transform_image(
        slice_img, output_size=list(invivo_img.shape)
    ))
    
    # Step 2: Save graph
    g.save()
    print(f"  Graph saved to: {GRAPH_PATH}")
    
    # Step 3: Reload graph
    g_check = ca.Graph.load(str(GRAPH_PATH))
    t_reloaded = g_check.get_transform(slice_name, invivo_node)
    slice_img_reload = g_check.get_image(slice_name)
    
    warped_reloaded = np.asarray(t_reloaded.transform_image(
        slice_img_reload, output_size=list(invivo_img.shape)
    ))
    
    # Step 4: Compare
    max_diff = np.max(np.abs(warped_original.astype(np.float64) - warped_reloaded.astype(np.float64)))
    mean_diff = np.mean(np.abs(warped_original.astype(np.float64) - warped_reloaded.astype(np.float64)))
    
    print(f"\n  Original transform:  {type(t_original).__name__}")
    print(f"  Reloaded transform:  {type(t_reloaded).__name__}")
    print(f"  Max absolute diff:   {max_diff:.6e}")
    print(f"  Mean absolute diff:  {mean_diff:.6e}")
    
    if max_diff < 1e-3:
        print(f"\n  PASS: Graph persistence verified")
    else:
        print(f"\n  WARNING: Max diff {max_diff:.6e} exceeds threshold 1e-3")
        print(f"  This may indicate SQLite serialization issues (check Samba mount)")
    
    del g_check, warped_original, warped_reloaded
    gc.collect()

---

## 14. GraphViewer: See All Aligned Slices

Visualize all aligned slices together in the in-vivo coordinate space.

In [ ]:
# =============================================================================
# GRAPHVIEWER: VISUALIZE ALL ALIGNED SLICES
# =============================================================================

print("="*60)
print("GRAPHVIEWER")
print("="*60)
print(f"Aligned slices: {len(aligned_slices)}")
print()

if len(aligned_slices) == 0:
    print("No slices aligned yet. Run an alignment mode first.")
else:
    if GraphViewer is None:
        # Fallback if GraphViewer not available
        print("GraphViewer not available in this version.")
        print("Using manual napari viewer instead...")
        print()
        
        import napari
        
        v = napari.Viewer()
        
        # Add in-vivo reference
        invivo_img = g.get_image(invivo_node)
        v.add_image(invivo_img, name=invivo_node, colormap="gray", opacity=0.7)
        
        # Add each aligned slice, transformed to invivo space
        colormaps = ["red", "green", "blue", "cyan", "magenta", "yellow"]
        
        for i, slice_name in enumerate(sorted(aligned_slices)):
            try:
                # Get slice and transform
                slice_img = g.get_image(slice_name)
                transform = g.get_transform(slice_name, invivo_node)
                
                # Apply transform
                transformed = transform.transform_image(slice_img)
                
                # Get origin if available
                origin = transformed.origin if hasattr(transformed, "origin") else [0, 0, 0]
                
                # Add to viewer
                cmap = colormaps[i % len(colormaps)]
                v.add_image(
                    transformed, 
                    name=slice_name, 
                    colormap=cmap, 
                    opacity=0.5,
                    translate=origin
                )
                print(f"  Added: {slice_name} ({cmap})")
            except Exception as e:
                print(f"  Failed: {slice_name} - {e}")
        
        print("\n✓ Viewer opened")
        napari.run()
        
    else:
        # Use GraphViewer
        print("Opening GraphViewer in invivo_ref space...")
        print()
        
        v = GraphViewer(g, space=invivo_node)
        
        # Add in-vivo reference
        v.add_image(invivo_node, colormap="gray", opacity=0.7)
        
        # Add aligned slices
        colormaps = ["red", "green", "blue", "cyan", "magenta", "yellow"]
        
        for i, slice_name in enumerate(sorted(aligned_slices)):
            try:
                cmap = colormaps[i % len(colormaps)]
                v.add_image(slice_name, colormap=cmap, opacity=0.5)
                print(f"  Added: {slice_name} ({cmap})")
            except Exception as e:
                print(f"  Failed: {slice_name} - {e}")
        
        print("\n✓ GraphViewer opened")

---

## 15. Alignment Status Summary

Quick overview of all slices and their alignment status, including transform types and Mode D alignments.

In [ ]:
# =============================================================================
# ALIGNMENT STATUS SUMMARY
# =============================================================================

# Refresh alignment status from graph
aligned_slices = []
unaligned_slices = []

for s in slice_nodes:
    if s in g.edges and invivo_node in g.edges[s]:
        aligned_slices.append(s)
    else:
        unaligned_slices.append(s)

print("="*60)
print("ALIGNMENT STATUS")
print("="*60)
print(f"Total slices: {len(slice_nodes)}")
print(f"Aligned: {len(aligned_slices)} ({100*len(aligned_slices)/len(slice_nodes):.1f}%)")
print(f"Remaining: {len(unaligned_slices)}")
print()

# Progress bar
progress = len(aligned_slices) / len(slice_nodes)
bar_width = 40
filled = int(bar_width * progress)
bar = "=" * filled + "-" * (bar_width - filled)
print(f"Progress: [{bar}] {100*progress:.1f}%")
print()

# List aligned slices with transform type
if aligned_slices:
    print("Aligned slices:")
    # Group by transform type
    transform_counts = {}
    for s in sorted(aligned_slices):
        t = g.get_transform(s, invivo_node)
        tname = type(t).__name__
        transform_counts[tname] = transform_counts.get(tname, 0) + 1
        print(f"  + {s}: {tname}")
    print()
    print("Transform summary:")
    for tname, count in sorted(transform_counts.items()):
        print(f"  {tname}: {count} slices")
    print()

# List next few unaligned
if unaligned_slices:
    print("Next unaligned slices:")
    for s in sorted(unaligned_slices)[:5]:
        print(f"  o {s}")
    if len(unaligned_slices) > 5:
        print(f"  ... and {len(unaligned_slices) - 5} more")
    print()

# Check for Mode D (same-dimension) alignments
try:
    if 'g_samedim' in dir() and g_samedim is not None:
        print("Mode D (same-dimension) alignments:")
        for e1 in g_samedim.edges:
            for e2 in g_samedim.edges[e1]:
                t = g_samedim.edges[e1][e2]
                print(f"  {e1} -> {e2}: {type(t).__name__}")
        print()
except NameError:
    pass

# Update dropdown
slice_dropdown.options = get_slice_options()

print("="*60)

---

## 16. Graph Structure Visualization

View the graph structure as a node-edge diagram.

In [ ]:
# =============================================================================
# GRAPH STRUCTURE VISUALIZATION
# =============================================================================

print(f"Graph: {g.name}")
print(f"Nodes: {len(g.nodes)}")
print(f"Edges: {sum(len(e) for e in g.edges.values()) // 2}")
print()

# Visualize (uses SVG via patch)
g.visualise()

---

## 17. Advanced: `alignment_gui()` with Crop & References

Power-user cell showing how to call `alignment_gui()` directly instead of `align_interactive()`.

**Trade-off:** `alignment_gui()` handles a single transform at a time (no composition loop like `align_interactive()`), but supports two extra parameters:
- **`crop=True`** — only renders the region of the movable image that overlaps the fixed image, saving memory
- **`references`** — list of additional images to display as alignment aids

**Note:** `crop` does NOT work with `align_interactive()` — it's only available through `alignment_gui()`.

In [ ]:
# =============================================================================
# ADVANCED: alignment_gui() with crop and references
# =============================================================================
# alignment_gui() is the lower-level function that align_interactive() calls
# internally. It opens a single napari window for one transform at a time.
#
# Advantages over align_interactive():
#   - crop=True reduces memory by only rendering the overlap region
#   - references=[] lets you show additional images as guides
#
# Disadvantages:
#   - No composition loop (t, T, R, N, etc.) — one transform per call
#   - No save-to-graph shortcuts (s/S) — you manage saving yourself
# =============================================================================

slice_name = get_selected_slice()
slice_img = g.get_image(slice_name)
invivo_img = g.get_image(invivo_node)

# Pad for point-based transforms
PAD_Z = 25
padded = load_and_pad_slice(slice_name, pad_z=PAD_Z)

# Get starting transform
prev_transform = get_previous_transform(slice_name)
if prev_transform is not None:
    initial_transform = prev_transform + ca.TranslateRotate  # Point-based rigid
else:
    initial_transform = get_initial_transform_3d_base() + ca.TranslateRotate

# Optional: add a reference image (e.g., a previously aligned slice)
# references = [(ref_img, ref_transform)]  # list of (image, transform) tuples
references = []

print("Launching alignment_gui() with crop=True")
print("This renders only the overlap region (lower memory)")
print()

transform = ca_gui.alignment_gui(
    movable_image=padded,
    base_image=invivo_img,
    transform=initial_transform,
    crop=True,
    references=references,
)

# Manual save
save_alignment(slice_name, transform)

del padded
gc.collect()

---

## 18. Advanced: Node-Name Mode

Instead of passing raw arrays to `align_interactive()`, you can pass **node names** (strings) and let CASTalign resolve images from the graph. This is simpler but doesn't work with padding (Modes A/C) since the padded array isn't stored as a graph node.

**Use when:** Both images are already stored in the graph as nodes (e.g., Mode D, or unpadded Mode B).

In [ ]:
# =============================================================================
# ADVANCED: NODE-NAME MODE
# =============================================================================
# When both images are stored as graph nodes, you can pass their names
# directly. CASTalign will call g.get_image() internally.
#
# This ONLY works when:
#   - Both nodes exist in the graph
#   - You don't need padding (the graph stores the raw image, not padded)
#   - The graph is passed via graph=g
#
# It does NOT work for Modes A/C where we pad the slice before alignment.
# =============================================================================

slice_name = get_selected_slice()
print(f"Node-name alignment: {slice_name} -> {invivo_node}")
print(f"(Images will be loaded from graph automatically)")
print()

# Get initial transform
prev_transform = get_previous_transform(slice_name)
initial_transform = prev_transform if prev_transform is not None else get_initial_transform_3d_base()

# align_interactive resolves node names to images via the graph
transform = ca_gui.align_interactive(
    nodes_movable=slice_name,    # String node name, not array
    nodes_fixed=invivo_node,     # String node name, not array
    transform=initial_transform,
    graph=g                      # Required when using node names
)

save_alignment(slice_name, transform)

---

## 19. Troubleshooting & Tips

### SQLite over Samba/SMB
- Graph `.db` files are SQLite databases. Writing over network mounts can cause "database is locked" or corruption errors.
- **Fix:** Copy the `.db` file locally, do alignment, copy back. Or run alignment directly on the server via SSH + X11 forwarding.
- Use the Graph Reload Verification cell (Cell 13) to confirm persistence after saving.

### Memory Management
- The in-vivo stack (~200 MB) stays in RAM once loaded by the graph. Padded slices add ~100 MB each.
- After alignment, `del padded; gc.collect()` frees the padded array.
- If napari becomes sluggish, restart the kernel and re-run from Cell 1.
- `crop=True` in `alignment_gui()` (Cell 17) reduces memory by only rendering the overlap region.

### Padding Rationale
- Point-based transforms (T, R, P, N, V) need at least a few z-slices to fit 3D correspondences.
- A 1-voxel slice has no z-extent, so the fitter can't determine z-position.
- We pad to `2*PAD_Z+1` slices (default 51) with the original at center (z=25).
- Padding value = 90% of slice mean to avoid confusing the fitter with zeros or max values.

### Label Images & `image_is_label`
- CASTalign auto-detects label images (integer dtype, few unique values) and uses nearest-neighbor interpolation.
- Cell masks and other continuous-valued images can be misdetected as labels.
- We disable this globally with `ca_utils.image_is_label = lambda img: False` in the imports cell.
- To re-enable for specific images, pass `labels=True` to `transform_image()`.

### Transform Composition & Inversion
- Transforms compose left-to-right: `t1 + t2` means "apply t1 first, then t2".
- The graph stores directed edges. `g.get_transform(A, B)` returns the A->B direction.
- CASTalign automatically inverts transforms when traversing edges in reverse.
- To manually invert: most affine transforms support `.invert()`, but nonlinear transforms (Triangulation) may only have numerical inversion.

### Common Issues
- **"Point-based transforms don't fit"** — Make sure you have at least 3 point pairs (4+ recommended). Check that points are on the correct layers (base vs movable).
- **"Transform looks wrong after loading"** — Run the Graph Reload Verification cell. If it fails, the SQLite file may be corrupted (Samba issue).
- **"GUI shows blank/black"** — The images may have very different intensity ranges. Try normalizing before alignment, or adjust napari contrast limits manually.
- **"Compose error: transform types incompatible"** — Some transforms can't be composed. Use `align_interactive()` which handles this via its composition loop.